In [10]:
!pip install metaflow kagglehub pandas scikit-learn -q
import os
import sys

# Metaflow environment setup
os.environ['USERNAME'] = 'sidra_saqlain'
os.environ['METAFLOW_DEFAULT_METADATA'] = 'local'

In [11]:
from metaflow import FlowSpec, step, Runner
import pandas as pd

class MetaflowNotebookRegistry:
    def __init__(self):
        self.steps = {}

    def register(self, name, func):
        self.steps[name] = func
        print(f"Step '{name}' registered.")

# Initialize engine
registry = MetaflowNotebookRegistry()

In [12]:
def start(self):
    import kagglehub
    path = kagglehub.dataset_download("shivamb/netflix-shows")
    csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
    self.df = pd.read_csv(os.path.join(path, csv_file))
    print(f"Data Loaded: {self.df.shape}")
    self.next(self.preprocess)

registry.register("start", start)

Step 'start' registered.


In [13]:
def preprocess(self):
    self.df = self.df.dropna(subset=['type'])
    self.df['is_movie'] = (self.df['type'] == 'Movie').astype(int)
    print("Preprocessing finished in a separate cell!")
    self.next(self.end)

registry.register("preprocess", preprocess)

Step 'preprocess' registered.


In [14]:
def end(self):
    print("Metaflow Run Complete!")

registry.register("end", end)

Step 'end' registered.


In [20]:
import sys
import os
import inspect
import subprocess

def run_notebook_native_flow():
    """
    Core engine to assemble individual notebook cells into a
    valid Metaflow FlowSpec class and execute it via a virtual script.
    """
    script_name = "dynamic_netflix_flow.py"

    # Define the Base Structure & Essential Global Imports
    flow_code = [
        "from metaflow import FlowSpec, step",
        "import pandas as pd",
        "import kagglehub",
        "import os",
        "from sklearn.preprocessing import LabelEncoder",
        "",
        "class NativeNetflixFlow(FlowSpec):"
    ]

    # Iterate through the Registry to assemble steps with correct indentation
    for name, func in registry.steps.items():
        # Extract the raw source code of the registered function
        lines = inspect.getsource(func).splitlines()

        # Inject Metaflow @step decorator
        flow_code.append(f"    @step")

        # Reconstruct the function definition line to match the class context
        flow_code.append(f"    def {name}(self):")

        # Apply standard Python indentation (8 spaces: 4 for Class + 4 for Method)
        for line in lines[1:]:
            # Clean original indentation and apply new structural indentation
            stripped_line = line.strip()
            if stripped_line:
                flow_code.append("        " + stripped_line)
            else:
                flow_code.append("")

        flow_code.append("") # Maintain spacing between steps

    # Append the main execution block
    flow_code.append("\nif __name__ == '__main__':\n    NativeNetflixFlow()")

    # Persist the dynamically generated code to a virtual script file
    with open(script_name, "w") as f:
        f.write("\n".join(flow_code))

    print(f"Status: Created Virtual Script -> {script_name}")
    print("Action: Initiating Metaflow Runner...")

    try:
        # Execute the generated script as a standalone process
        result = subprocess.run([sys.executable, script_name, "run"], capture_output=True, text=True)

        if result.returncode == 0:
            print("\nCOMPLIANCE CHECK: Metaflow multi-cell execution successful.")
            print(result.stdout)
        else:
            print("\nCRITICAL: Metaflow Execution Error Detected:")
            print(result.stderr)
            # Log the generated snippet for debugging purposes
            print("\n--- Source Snippet for Inspection ---")
            print("\n".join(flow_code[7:20]))

    except Exception as e:
        print(f"System Failure: {e}")

# Execute the orchestration
run_notebook_native_flow()

Status: Created Virtual Script -> dynamic_netflix_flow.py
Action: Initiating Metaflow Runner...

COMPLIANCE CHECK: Metaflow multi-cell execution successful.
2026-03-25 17:24:08.429 Workflow starting (run-id 1774459448427437):
2026-03-25 17:24:08.448 [1774459448427437/start/1 (pid 4013)] Task is starting.
2026-03-25 17:24:11.495 [1774459448427437/start/1 (pid 4013)] Using Colab cache for faster access to the 'netflix-shows' dataset.
2026-03-25 17:24:11.577 [1774459448427437/start/1 (pid 4013)] Data Loaded: (8807, 12)
2026-03-25 17:24:12.094 [1774459448427437/start/1 (pid 4013)] Task finished successfully.
2026-03-25 17:24:12.100 [1774459448427437/preprocess/2 (pid 4034)] Task is starting.
2026-03-25 17:24:14.787 [1774459448427437/preprocess/2 (pid 4034)] Preprocessing finished in a separate cell!
2026-03-25 17:24:15.412 [1774459448427437/preprocess/2 (pid 4034)] Task finished successfully.
2026-03-25 17:24:15.420 [1774459448427437/end/3 (pid 4055)] Task is starting.
2026-03-25 17:24:18.